In [1]:
import sys
sys.path.insert(1, '../Codes_1Dtree')
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"

Load libraries

In [2]:
import torch
from torch_geometric.loader import DataLoader
from data.graph_dataset import OneDDatasetBuilder, OneDDatasetLoader, normalize, dataset_to_loader
# from networks.gcn import GraphUNet, RecurrentFormulationNet
from networks.gcnv2 import RecurrentFormulationNet
import matplotlib.pyplot as plt

from data.graph_dataset import batchgraph_generation_wise

In [3]:
class objectview(object):
    def __init__(self, d) -> None:
        self.__dict__ = d
    def setattr(self, attr_name, attr_value):
        self.__dict__[attr_name] = attr_value

args = objectview({
    'n_field': 2,
    'n_meshfield': 18,
    'hidden_size': 128,
    'latent_size': 128,
    'aggr': 'sum',
    'act': 'mish',
    'alpha': 0.5,
    'device': torch.device('cuda:0' if torch.cuda.is_available() else 'cpu'),
    'lr': 5e-7,
    'weight_decay': 5e-3,
    'n_epoch': 200,
    'timestep': None,
    'timeslice_hops': 5,
    'timeslice_steps': 1,
    'n_data_per_batch': 1,
    'criterion': torch.nn.MSELoss(),
    'plot': False
})

Load dataset

In [4]:
# dataset = OneDDatasetBuilder(
#     raw_dir='/data1/tam/datasets_231228',
#     root_dir='/data1/tam/downloaded_datasets_WT_v2',
#     sub_dir='processed',
#     subjects='all',
#     time_names=[str(i).zfill(3) for i in range(201)],
#     data_type = torch.float32,
#     readme='edge_index(2xn_edge), node_attr(n_nodex18), pressure+flowrate(n_nodex201), pressure_dot+flowrate_dot(n_nodex201)'
# )
# dataset = OneDDatasetLoader(
#     root_dir='/data1/tam/downloaded_datasets_WT_v2',
#     sub_dir='processed',
#     subjects='all',
#     time_names=[str(i).zfill(3) for i in range(201)],
#     data_type = torch.float32
# )
# _x = torch.clip(dataset[0].flowrate.flatten(), -1e-6, 1e-6)
# if args.plot:
#     plt.xlim([-1,1])
#     plt.hist(_x, bins=100)
#     plt.show()

In [5]:
# dataset = normalize(
#     dataset=dataset,
#     sub_dir='normalized',
#     scaler_dict={
#         'node_attr': ('minmax_scaler', 0, None),
#         'pressure': ('minmax_scaler', None, None),
#         'flowrate': ('minmax_scaler', None, (-1e-6, 1e-6)),
#         'pressure_dot': ('minmax_scaler', None, None),
#         'flowrate_dot': ('minmax_scaler', None, (-1e-6, 1e-6))
#     }
# )
# dataset = OneDDatasetLoader(
#     root_dir='/data1/tam/downloaded_datasets_WT_v2',
#     sub_dir='normalized',
#     subjects='all',
#     time_names=[str(i).zfill(3) for i in range(201)],
#     data_type = torch.float32
# )
# _x = dataset[0].flowrate.flatten()
# if args.plot:
#     plt.xlim([-1,1])
#     plt.hist(_x, bins=100)
#     plt.show()

In [6]:
# dataset = batchgraph_generation_wise(
#     dataset,
#     sub_dir='batched',
#     batch_gens=[[0,9],[10,13],[14,17],[17,50]],
#     timestep=args.timestep,
#     timeslice_hops=args.timeslice_hops,
#     timeslice_steps=args.timeslice_steps
# )
dataset = OneDDatasetLoader(
    root_dir='/data1/tam/downloaded_datasets_WT_v2',
    sub_dir='batched',
    subjects='all',
    time_names=[str(i).zfill(3) for i in range(201)],
    data_type = torch.float32
)
if args.plot:
    print(dataset[0])

In [7]:
(train_loader, test_loader) = dataset_to_loader(
    dataset=dataset,
    data_subset_dict={
        'train': list(range(0, 30)),
        'test': list(range(30, 35))
    },
    n_data_per_batch=args.n_data_per_batch
)

Model initializing and training

In [8]:
model = RecurrentFormulationNet(
    n_field=args.n_field,
    n_meshfield=args.n_meshfield,
    hidden_size=args.hidden_size,
    latent_size=args.latent_size,
    act=args.act
)
setattr(model, 'name', 'PARC_GCN_UNet_full')
model = model.to(args.device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
setattr(args, 'optimizer', optimizer)

In [9]:
def train(model, data, args):
    ##
    F_true = torch.cat([data.pressure.unsqueeze(2), data.flowrate.unsqueeze(2)], dim=2) \
                .float().to(args.device)
    F_dot_true = torch.cat([data.pressure_dot.unsqueeze(2), data.flowrate_dot.unsqueeze(2)], dim=2) \
                .float().to(args.device)
    ##
    F_0 = F_true[:,0,:]
    edge_index = data.edge_index.to(args.device)
    node_attr = data.node_attr.float().to(args.device)
    F_true = F_true[:,1:,:]
    F_dot_true = F_dot_true[:,1:,:]
    ##
    F_pred, F_dot_pred = model.forward(
        F_0=F_0,
        edge_index=edge_index,
        meshfield=node_attr,
        n_time=data.number_of_timesteps - 1
    )
    ##
    loss = args.criterion(F_pred, F_true)*args.alpha + (1.-args.alpha)*args.criterion(F_dot_pred, F_dot_true)
    loss.backward()
    args.optimizer.step()
    
    return loss.item()

def eval(model, data, args):
    ##
    F_true = torch.cat([data.pressure.unsqueeze(2), data.flowrate.unsqueeze(2)], dim=2) \
                .float().to(args.device)
    F_dot_true = torch.cat([data.pressure_dot.unsqueeze(2), data.flowrate_dot.unsqueeze(2)], dim=2) \
                .float().to(args.device)
    ##
    F_0 = F_true[:,0,:]
    edge_index = data.edge_index.to(args.device)
    node_attr = data.node_attr.float().to(args.device)
    F_true = F_true[:,1:,:]
    F_dot_true = F_dot_true[:,1:,:]
    ##
    with torch.no_grad():
        F_pred, F_dot_pred = model.forward(
            F_0=F_0,
            edge_index=edge_index,
            meshfield=node_attr,
            n_time=data.number_of_timesteps - 1
        )
        loss = args.criterion(F_pred, F_true)*args.alpha + (1.-args.alpha)*args.criterion(F_dot_pred, F_dot_true)
        
    return loss.item()

In [10]:
# Training
total_train_loss = []
total_eval_loss = []
for epoch in range(args.n_epoch):
    torch.cuda.empty_cache()
    train_loss = 0
    for i in range(train_loader.__len__()):
        data = next(iter(train_loader))
        train_loss += train(model=model, data=data, args=args)
    train_loss /= train_loader.__len__()
    total_train_loss.append(train_loss)

    eval_loss = 0
    for i in range(test_loader.__len__()):
        data = next(iter(test_loader))
        eval_loss += eval(model=model, data=data, args=args)
    eval_loss /= test_loader.__len__()
    total_eval_loss.append(eval_loss)
    
    print(f'Epoch {epoch}: train loss = {train_loss}; eval loss = {eval_loss}')
    if (epoch+1) % 20 == 0:
        torch.save(model.state_dict(), f'models/{model.name}_node0_epoch{epoch+1}.pth')

/home/mlfm/anaconda3/envs/geometric/lib/python3.11/site-packages/torch_geometric/utils/sparse.py:176: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:54.)
  return adj.to_sparse_csr()


Epoch 0: train loss = 0.7557681789000829; eval loss = 0.5634023070335388
Epoch 1: train loss = 0.6585306818286578; eval loss = 0.509990268945694
Epoch 2: train loss = 0.6181084300080936; eval loss = 0.4782132342457771
Epoch 3: train loss = 0.585323832432429; eval loss = 0.45155143886804583
Epoch 4: train loss = 0.5303144743045171; eval loss = 0.4277245044708252
Epoch 5: train loss = 0.4885794724027316; eval loss = 0.40423655658960345
Epoch 6: train loss = 0.4725010690589746; eval loss = 0.40460274517536166
Epoch 7: train loss = 0.4647276495893796; eval loss = 0.40781288743019106
Epoch 8: train loss = 0.44796214550733565; eval loss = 0.40471420288085935
Epoch 9: train loss = 0.4407805636525154; eval loss = 0.40569558590650556
Epoch 10: train loss = 0.4354294776916504; eval loss = 0.4033400923013687
Epoch 11: train loss = 0.41678520838419597; eval loss = 0.40277906507253647
Epoch 12: train loss = 0.4026371665298939; eval loss = 0.39965904504060745
Epoch 13: train loss = 0.394842524329821